# Module 9 • Machine Translation

# Lesson 54 • Arabic–English Machine Translation — Morphology, Tokenization, and Diacritization

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Execution target:** CPU only

---

## Scope

This lesson focuses specifically on Arabic↔English machine translation.

It covers:

- Arabic morphological richness;
- clitics and affixes;
- tokenization choices;
- word-level processing;
- subword-aware processing;
- character-level processing;
- diacritization and tashkeel preservation;
- lexical ambiguity;
- translation directionality;
- morphology-sensitive evaluation;
- Arabic-specific error analysis.

The executable core is fully offline and compares simple representations and
translation-oriented preprocessing choices on a small Arabic–English corpus.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain why Arabic morphology complicates MT;
- identify common Arabic clitics;
- distinguish word-, subword-, and character-level tokenization;
- preserve tashkeel correctly in fully vocalized experiments;
- measure vocabulary fragmentation;
- analyze tokenization efficiency;
- compare Arabic→English and English→Arabic challenges;
- identify lexical ambiguity caused by missing diacritics;
- design morphology-sensitive MT evaluation;
- build a reproducible Arabic–English MT preprocessing pipeline.

## Table of Contents

1. Why Arabic MT Is Difficult  
2. Morphological Richness  
3. Clitics  
4. Inflection and Derivation  
5. Orthographic Variation  
6. Tashkeel  
7. Ambiguity Without Tashkeel  
8. Arabic→English Direction  
9. English→Arabic Direction  
10. Word-Level Tokenization  
11. Subword Tokenization  
12. Character-Level Tokenization  
13. Offline Parallel Corpus  
14. Arabic Unicode Inspection  
15. Tashkeel Detection  
16. Word Tokenizer  
17. Character Tokenizer  
18. Simple Clitic Segmentation  
19. Vocabulary Statistics  
20. Sequence-Length Comparison  
21. OOV Analysis  
22. Morphology-Aware Representation  
23. Tokenization Efficiency  
24. Translation Lexicon Baseline  
25. Word-Level Translation Baseline  
26. Character-Aware Backoff  
27. Direction-Specific Translation  
28. Lexical Ambiguity  
29. Diacritization as Disambiguation  
30. Tashkeel Preservation Check  
31. Evaluation Dataset  
32. Exact Match  
33. Token Accuracy  
34. chrF-Like Character Score  
35. Morphology-Sensitive Error Categories  
36. Diacritization Errors  
37. Clitic Errors  
38. Agreement Errors  
39. Named Entities  
40. Tokenization Ablation  
41. Character-Level Trade-Offs  
42. Subword Trade-Offs  
43. Pretrained Multilingual MT Considerations  
44. Data Preparation Checklist  
45. Reproducibility  
46. Knowledge Check  
47. Exercises  
48. Summary and Next Lesson

# 1. Why Arabic MT Is Difficult

Arabic is morphologically rich. A single orthographic token can encode information
that may require several English words.

Example:

```text
وَسَيَكْتُبُونَهَا
```

may contain conjunction, future marking, verb morphology, subject agreement, and
an object pronoun within one surface form.

In [ ]:
import platform
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

challenge_table = pd.DataFrame(
    [
        ("Morphology", "many surface forms from one lexical base"),
        ("Clitics", "multiple grammatical elements attached to words"),
        ("Tashkeel", "diacritics can disambiguate pronunciation and meaning"),
        ("Orthography", "multiple written variants may occur"),
        ("Directionality", "Arabic→English and English→Arabic behave differently"),
    ],
    columns=["Challenge", "Effect on MT"],
)

challenge_table

# 2. Morphological Richness

Arabic combines:

- prefixes;
- stems;
- inflectional suffixes;
- pronominal enclitics.

This increases vocabulary sparsity for word-level systems.

# 3. Clitics

Frequent proclitic categories include conjunctions and prepositions.

Examples:

- `وَ` — and;
- `فَ` — so/then;
- `بِ` — with/by;
- `لِ` — for/to;
- `كَ` — like/as.

Enclitics may encode object or possessive pronouns.

# 4. Inflection and Derivation

Arabic morphology carries:

- person;
- number;
- gender;
- tense/aspect;
- definiteness;
- case in fully vocalized text.

Translation systems must learn which distinctions need explicit expression in the
target language.

# 5. Orthographic Variation

Arabic text may vary in:

- hamza forms;
- alif variants;
- ya/alif maqsura;
- ta marbuta;
- spacing;
- punctuation.

Normalization decisions must match the research question.

# 6. Tashkeel

Tashkeel includes marks such as:

- fatḥa;
- ḍamma;
- kasra;
- sukūn;
- shadda;
- tanwīn.

In fully vocalized MT, tashkeel is part of the data and must be preserved.

# 7. Ambiguity Without Tashkeel

Removing diacritics can collapse distinct readings into the same consonantal
skeleton.

This can increase lexical ambiguity and harm translation when the context is
insufficient.

# 8. Arabic→English Direction

Arabic→English often requires:

- decomposing morphology;
- resolving clitics;
- selecting English function words;
- mapping rich inflection into a less morphologically rich target.

# 9. English→Arabic Direction

English→Arabic often requires the model to generate:

- gender agreement;
- number agreement;
- inflection;
- clitic attachment;
- potentially tashkeel.

Target-side generation is therefore morphologically demanding.

# 10. Word-Level Tokenization

Word-level tokenization is intuitive but suffers from high sparsity.

# 11. Subword Tokenization

Subword methods split words into frequent units and reduce out-of-vocabulary
problems.

Examples include BPE, SentencePiece, and unigram language-model tokenization.

# 12. Character-Level Tokenization

Character-level models remove word-level OOV problems and preserve fine
orthographic detail.

Their main cost is much longer sequences.

# 13. Offline Parallel Corpus

In [ ]:
parallel_pairs = [
    ("أَنَا أَكْتُبُ الْكِتَابَ", "i write the book"),
    ("هُوَ يَكْتُبُ الْكِتَابَ", "he writes the book"),
    ("هِيَ تَقْرَأُ الْكِتَابَ", "she reads the book"),
    ("نَحْنُ نَقْرَأُ الْكِتَابَ", "we read the book"),
    ("ذَهَبْتُ إِلَى الْمَدْرَسَةِ", "i went to the school"),
    ("هُوَ فِي الْمَدْرَسَةِ", "he is in the school"),
    ("هِيَ فِي الْمَدْرَسَةِ", "she is in the school"),
    ("كِتَابُهُ جَدِيدٌ", "his book is new"),
    ("كِتَابُهَا جَدِيدٌ", "her book is new"),
    ("وَسَيَكْتُبُونَهَا", "and they will write it"),
    ("بِالْمَدْرَسَةِ", "at the school"),
    ("كِتَابُهُمَا", "their book"),
]

corpus = pd.DataFrame(
    parallel_pairs,
    columns=["Arabic", "English"],
)

corpus

# 14. Arabic Unicode Inspection

In [ ]:
sample = "وَسَيَكْتُبُونَهَا"

unicode_rows = []

for character in sample:
    unicode_rows.append(
        {
            "character": character,
            "codepoint": f"U+{ord(character):04X}",
        }
    )

pd.DataFrame(
    unicode_rows
)

# 15. Tashkeel Detection

In [ ]:
ARABIC_DIACRITICS = set(
    "\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652"
)

def contains_tashkeel(text):
    return any(
        character in ARABIC_DIACRITICS
        for character in text
    )

def tashkeel_count(text):
    return sum(
        character in ARABIC_DIACRITICS
        for character in text
    )

pd.Series({
    "contains_tashkeel": contains_tashkeel(sample),
    "tashkeel_count": tashkeel_count(sample),
})

# 16. Word Tokenizer

In [ ]:
def word_tokenize(text):
    return text.split()

word_tokenize(
    "أَنَا أَكْتُبُ الْكِتَابَ"
)

# 17. Character Tokenizer

Spaces can be retained explicitly or excluded.

Here we exclude spaces for basic sequence-length comparison.

In [ ]:
def character_tokenize(
    text,
    keep_spaces=False,
):
    if keep_spaces:
        return list(text)

    return [
        character
        for character in text
        if not character.isspace()
    ]

character_tokenize(
    "كِتَابُهَا"
)

# 18. Simple Clitic Segmentation

This educational segmenter handles only a few known fully vocalized prefixes and
suffixes.

It is not a complete Arabic morphological analyzer.

In [ ]:
PROCLITICS = [
    "وَ",
    "فَ",
    "بِ",
    "لِ",
    "كَ",
]

ENCLITICS = [
    "هَا",
    "هُ",
    "هُمَا",
    "هُمْ",
    "نَا",
]

def segment_simple_clitics(token):
    pieces = []
    remaining = token

    changed = True

    while changed:
        changed = False

        for prefix in PROCLITICS:
            if (
                remaining.startswith(prefix)
                and len(remaining) > len(prefix)
            ):
                pieces.append(prefix)
                remaining = remaining[
                    len(prefix):
                ]
                changed = True
                break

    suffix_piece = None

    for suffix in sorted(
        ENCLITICS,
        key=len,
        reverse=True,
    ):
        if (
            remaining.endswith(suffix)
            and len(remaining) > len(suffix)
        ):
            suffix_piece = suffix
            remaining = remaining[
                :-len(suffix)
            ]
            break

    if remaining:
        pieces.append(remaining)

    if suffix_piece:
        pieces.append(
            suffix_piece
        )

    return pieces

segmentation_examples = pd.DataFrame(
    [
        {
            "token": token,
            "segments": " + ".join(
                segment_simple_clitics(
                    token
                )
            ),
        }
        for token in [
            "وَسَيَكْتُبُونَهَا",
            "بِالْمَدْرَسَةِ",
            "كِتَابُهَا",
        ]
    ]
)

segmentation_examples

# 19. Vocabulary Statistics

In [ ]:
arabic_word_tokens = [
    token
    for sentence in corpus["Arabic"]
    for token in word_tokenize(sentence)
]

english_word_tokens = [
    token
    for sentence in corpus["English"]
    for token in word_tokenize(sentence)
]

arabic_char_tokens = [
    token
    for sentence in corpus["Arabic"]
    for token in character_tokenize(sentence)
]

stats = pd.Series({
    "Arabic word types": len(set(arabic_word_tokens)),
    "Arabic word tokens": len(arabic_word_tokens),
    "Arabic character types": len(set(arabic_char_tokens)),
    "Arabic character tokens": len(arabic_char_tokens),
    "English word types": len(set(english_word_tokens)),
})

stats

# 20. Sequence-Length Comparison

In [ ]:
length_rows = []

for arabic, english in parallel_pairs:
    length_rows.append(
        {
            "Arabic": arabic,
            "word_tokens": len(
                word_tokenize(arabic)
            ),
            "character_tokens": len(
                character_tokenize(arabic)
            ),
            "English_words": len(
                word_tokenize(english)
            ),
        }
    )

length_frame = pd.DataFrame(
    length_rows
)

length_frame

In [ ]:
plt.figure(
    figsize=(7, 5)
)

plt.scatter(
    length_frame["word_tokens"],
    length_frame["character_tokens"],
)

plt.xlabel(
    "Arabic word-token count"
)

plt.ylabel(
    "Arabic character-token count"
)

plt.title(
    "Word vs Character Sequence Length"
)

plt.tight_layout()
plt.show()

# 21. OOV Analysis

Word-level systems can encounter unseen surface forms even when their components
have been observed.

In [ ]:
train_pairs = parallel_pairs[:9]
test_pairs = parallel_pairs[9:]

train_arabic_words = {
    token
    for arabic, _
    in train_pairs
    for token in word_tokenize(
        arabic
    )
}

oov_rows = []

for arabic, english in test_pairs:
    tokens = word_tokenize(
        arabic
    )

    oov = [
        token
        for token in tokens
        if token not in train_arabic_words
    ]

    oov_rows.append(
        {
            "Arabic": arabic,
            "English": english,
            "OOV words": oov,
            "OOV count": len(oov),
        }
    )

pd.DataFrame(
    oov_rows
)

# 22. Morphology-Aware Representation

A morphology-aware representation can reduce sparsity by separating clitics from a
larger orthographic token.

In [ ]:
def morphology_aware_sentence(
    sentence,
):
    pieces = []

    for token in word_tokenize(
        sentence
    ):
        pieces.extend(
            segment_simple_clitics(
                token
            )
        )

    return pieces

morphology_aware_sentence(
    "وَسَيَكْتُبُونَهَا"
)

# 23. Tokenization Efficiency

In [ ]:
efficiency_rows = []

for arabic, _ in parallel_pairs:
    word_count = len(
        word_tokenize(
            arabic
        )
    )

    morph_count = len(
        morphology_aware_sentence(
            arabic
        )
    )

    char_count = len(
        character_tokenize(
            arabic
        )
    )

    efficiency_rows.append(
        {
            "Arabic": arabic,
            "word": word_count,
            "morphology_aware": (
                morph_count
            ),
            "character": char_count,
        }
    )

efficiency_frame = pd.DataFrame(
    efficiency_rows
)

efficiency_frame

# 24. Translation Lexicon Baseline

We build a tiny manually aligned lexical dictionary for illustration.

In [ ]:
arabic_to_english = {
    "أَنَا": "i",
    "هُوَ": "he",
    "هِيَ": "she",
    "نَحْنُ": "we",
    "أَكْتُبُ": "write",
    "يَكْتُبُ": "writes",
    "تَقْرَأُ": "reads",
    "نَقْرَأُ": "read",
    "الْكِتَابَ": "the book",
    "الْمَدْرَسَةِ": "the school",
    "جَدِيدٌ": "new",
    "إِلَى": "to",
    "فِي": "in",
}

english_to_arabic = {
    value: key
    for key, value
    in arabic_to_english.items()
}

# 25. Word-Level Translation Baseline

In [ ]:
def lexical_translate_ar_to_en(
    sentence,
):
    output = []

    for token in word_tokenize(
        sentence
    ):
        output.append(
            arabic_to_english.get(
                token,
                f"<UNK:{token}>",
            )
        )

    return " ".join(
        output
    )

lexical_translate_ar_to_en(
    "أَنَا أَكْتُبُ الْكِتَابَ"
)

# 26. Character-Aware Backoff

Character-aware systems can represent unseen forms even when a word dictionary
cannot translate them directly.

Here we simply expose the character decomposition for unknown items.

In [ ]:
def character_backoff(
    token,
):
    if token in arabic_to_english:
        return arabic_to_english[
            token
        ]

    return "|".join(
        character_tokenize(
            token
        )
    )

character_backoff(
    "وَسَيَكْتُبُونَهَا"
)

# 27. Direction-Specific Translation

Translation direction matters because target-language generation determines which
morphology must be produced.

In [ ]:
direction_table = pd.DataFrame(
    [
        (
            "Arabic→English",
            "analyze rich morphology",
            "generate comparatively lighter morphology",
        ),
        (
            "English→Arabic",
            "infer grammatical features",
            "generate rich morphology and clitics",
        ),
    ],
    columns=[
        "Direction",
        "Primary source-side challenge",
        "Primary target-side challenge",
    ],
)

direction_table

# 28. Lexical Ambiguity

Arabic consonantal forms can correspond to multiple analyses.

In real MT systems, context and diacritization may both contribute to
disambiguation.

In [ ]:
ambiguity_examples = pd.DataFrame(
    [
        (
            "علم",
            "can correspond to multiple readings depending on context",
        ),
        (
            "كتب",
            "can represent forms related to writing/books depending on analysis",
        ),
    ],
    columns=[
        "Unvocalized form",
        "Ambiguity note",
    ],
)

ambiguity_examples

# 29. Diacritization as Disambiguation

Tashkeel can encode distinctions that are invisible in unvocalized text.

Therefore, a diacritization-aware MT experiment should evaluate whether preserving
diacritics improves lexical disambiguation and target selection.

# 30. Tashkeel Preservation Check

In [ ]:
def extract_tashkeel(
    text,
):
    return "".join(
        character
        for character in text
        if character in ARABIC_DIACRITICS
    )

preservation_rows = []

for arabic, _ in parallel_pairs:
    preservation_rows.append(
        {
            "Arabic": arabic,
            "tashkeel_count": tashkeel_count(
                arabic
            ),
            "has_tashkeel": contains_tashkeel(
                arabic
            ),
        }
    )

pd.DataFrame(
    preservation_rows
)

# 31. Evaluation Dataset

In [ ]:
evaluation_pairs = [
    (
        "أَنَا أَكْتُبُ الْكِتَابَ",
        "i write the book",
    ),
    (
        "هُوَ يَكْتُبُ الْكِتَابَ",
        "he writes the book",
    ),
    (
        "هِيَ فِي الْمَدْرَسَةِ",
        "she is in the school",
    ),
]

pd.DataFrame(
    evaluation_pairs,
    columns=[
        "Arabic",
        "Reference",
    ],
)

# 32. Exact Match

In [ ]:
exact_rows = []

for arabic, reference in (
    evaluation_pairs
):
    prediction = (
        lexical_translate_ar_to_en(
            arabic
        )
    )

    exact_rows.append(
        {
            "Arabic": arabic,
            "reference": reference,
            "prediction": prediction,
            "exact": (
                prediction
                == reference
            ),
        }
    )

exact_frame = pd.DataFrame(
    exact_rows
)

exact_frame

# 33. Token Accuracy

In [ ]:
def token_accuracy(
    reference,
    prediction,
):
    reference_tokens = (
        reference.split()
    )

    prediction_tokens = (
        prediction.split()
    )

    if not reference_tokens:
        return 0.0

    matches = sum(
        reference_token
        == prediction_token
        for reference_token, prediction_token
        in zip(
            reference_tokens,
            prediction_tokens,
        )
    )

    return (
        matches
        / len(
            reference_tokens
        )
    )

token_accuracy(
    "i write the book",
    lexical_translate_ar_to_en(
        "أَنَا أَكْتُبُ الْكِتَابَ"
    ),
)

# 34. chrF-Like Character Score

chrF is a character n-gram metric. Here we implement a simplified character
unigram F1 to illustrate why character-level metrics are useful for morphology.

In [ ]:
def character_f1(
    reference,
    prediction,
):
    reference_chars = [
        c
        for c in reference
        if not c.isspace()
    ]

    prediction_chars = [
        c
        for c in prediction
        if not c.isspace()
    ]

    if (
        not reference_chars
        or not prediction_chars
    ):
        return 0.0

    reference_counts = Counter(
        reference_chars
    )

    prediction_counts = Counter(
        prediction_chars
    )

    overlap = sum(
        (
            reference_counts
            & prediction_counts
        ).values()
    )

    precision = (
        overlap
        / len(
            prediction_chars
        )
    )

    recall = (
        overlap
        / len(
            reference_chars
        )
    )

    if (
        precision
        + recall
        == 0
    ):
        return 0.0

    return (
        2
        * precision
        * recall
        / (
            precision
            + recall
        )
    )

character_f1(
    "i write the book",
    "i write the book",
)

# 35. Morphology-Sensitive Error Categories

In [ ]:
error_categories = pd.DataFrame(
    [
        ("Clitic", "attached function element translated incorrectly"),
        ("Inflection", "person/number/gender form wrong"),
        ("Agreement", "target agreement incorrect"),
        ("Tashkeel", "diacritics missing or incorrect"),
        ("Lexical ambiguity", "wrong sense selected"),
        ("Segmentation", "token boundaries harmful"),
        ("Word order", "target sequence reordered incorrectly"),
    ],
    columns=[
        "Error category",
        "Description",
    ],
)

error_categories

# 36. Diacritization Errors

In fully vocalized Arabic generation, evaluation should separately track:

- missing diacritics;
- incorrect diacritics;
- partially vocalized output.

# 37. Clitic Errors

Clitic errors can change both syntax and meaning.

Example categories:

- missing conjunction;
- wrong preposition;
- lost pronoun;
- incorrect attachment.

# 38. Agreement Errors

English→Arabic systems must generate gender, number, and person agreement that may
not be overtly marked in English.

# 39. Named Entities

Arabic↔English MT may require transliteration rather than translation for names.

Named-entity evaluation should distinguish:

- correct translation;
- correct transliteration;
- incorrect normalization.

# 40. Tokenization Ablation

We compare average sequence lengths across tokenization strategies.

In [ ]:
tokenization_summary = pd.Series({
    "Mean word length": (
        efficiency_frame["word"].mean()
    ),
    "Mean morphology-aware length": (
        efficiency_frame[
            "morphology_aware"
        ].mean()
    ),
    "Mean character length": (
        efficiency_frame[
            "character"
        ].mean()
    ),
})

tokenization_summary

In [ ]:
plt.figure(
    figsize=(7, 5)
)

plt.bar(
    tokenization_summary.index,
    tokenization_summary.values,
)

plt.ylabel(
    "Average tokens per sentence"
)

plt.title(
    "Arabic Tokenization Sequence Length"
)

plt.xticks(
    rotation=25
)

plt.tight_layout()
plt.show()

# 41. Character-Level Trade-Offs

Advantages:

- no word OOV;
- exact orthographic detail;
- tashkeel preserved naturally.

Disadvantages:

- much longer sequences;
- slower attention;
- harder long-range composition.

# 42. Subword Trade-Offs

Advantages:

- shorter than characters;
- lower sparsity than words;
- compatible with pretrained models.

Limitations:

- segmentation may not align with linguistic morphemes;
- tokenizer vocabulary can bias multilingual performance.

# 43. Pretrained Multilingual MT Considerations

When using MarianMT, M2M-100, NLLB, or similar models for Arabic:

- verify the exact language code;
- inspect tokenizer segmentation;
- test both translation directions independently;
- measure tashkeel preservation if the input is vocalized;
- evaluate morphology and named entities manually.

# 44. Data Preparation Checklist

For an Arabic↔English MT experiment:

1. define whether Arabic is vocalized or unvocalized;
2. keep train/dev/test preprocessing identical;
3. preserve alignment between parallel sentences;
4. choose tokenization deliberately;
5. report normalization rules;
6. train each direction separately;
7. report multiple metrics;
8. perform manual error analysis.

# 45. Reproducibility

In [ ]:
reproducibility = pd.Series(
    {
        "module": (
            "Module 9 • Machine Translation"
        ),
        "lesson": (
            "Lesson 54 • Arabic–English Machine Translation"
        ),
        "sentence_pairs": len(
            parallel_pairs
        ),
        "fully_vocalized_examples": True,
        "word_tokenizer": "whitespace",
        "character_tokenizer": (
            "Unicode characters excluding spaces"
        ),
        "simple_clitic_segmenter": True,
        "seed": SEED,
        "offline_execution": True,
        "python": (
            platform.python_version()
        ),
    },
    name="Lesson 54 experiment",
)

reproducibility

# 46. Knowledge Check

1. Why does Arabic morphology increase MT sparsity?
2. What are proclitics and enclitics?
3. Why can word-level tokenization be problematic?
4. How do subwords reduce OOV problems?
5. What is the main cost of character-level tokenization?
6. Why can tashkeel reduce ambiguity?
7. Why should fully vocalized experiments preserve tashkeel?
8. Why is Arabic→English different from English→Arabic?
9. What kinds of agreement errors occur in English→Arabic?
10. Why is chrF useful for morphologically rich languages?
11. What should tokenization ablations measure?
12. Why should MT directions be evaluated independently?
13. What role do named entities play?
14. Why is preprocessing consistency critical?
15. Why is manual error analysis still necessary?

# 47. Exercises

## Exercise 1
Add more fully vocalized Arabic–English sentence pairs.

## Exercise 2
Expand the clitic inventory.

## Exercise 3
Build an unvocalized version of the corpus and compare vocabulary size.

## Exercise 4
Measure tashkeel retention after tokenization.

## Exercise 5
Compare word, morphology-aware, and character sequence lengths.

## Exercise 6
Build a simple BPE tokenizer.

## Exercise 7
Compare Arabic→English and English→Arabic error categories.

## Exercise 8
Create a lexical-ambiguity test set.

## Exercise 9
Evaluate exact match and character F1 on vocalized Arabic outputs.

## Exercise 10
Design a full Arabic MT experiment with train/dev/test splits and multiple metrics.

## Challenge Exercises

1. Implement a real SentencePiece tokenizer.
2. Train a character-level Transformer for Arabic↔English.
3. Compare vocalized versus unvocalized Arabic translation.
4. Add morphology-aware statistical significance analysis.
5. Compare MarianMT, M2M-100, and NLLB on the same Arabic test set.

# 48. Summary and Next Lesson

In this lesson:

- Arabic morphological richness was connected directly to MT difficulty;
- proclitics, enclitics, inflection, derivation, orthography, and tashkeel were
  reviewed;
- word-, morphology-aware-, and character-level tokenization were compared;
- vocabulary size, OOV behavior, and sequence length were measured;
- a simple morphology-aware segmenter and lexical translation baseline were built;
- Arabic→English and English→Arabic were treated as distinct directions;
- lexical ambiguity, diacritization, clitic errors, agreement, named entities,
  and morphology-sensitive evaluation were covered;
- character-level and subword trade-offs were compared.

## Next Lesson

**Lesson 55: Machine Translation Evaluation — BLEU, chrF, COMET, Semantic
Similarity, and Statistical Significance** focuses on rigorous MT evaluation,
confidence intervals, standard deviation, bootstrap analysis, significance testing,
and system comparison.

# References

- Habash, N. *Introduction to Arabic Natural Language Processing*.
- Koehn, P. *Neural Machine Translation*.
- Sennrich, R. et al. work on subword neural machine translation.
- Papineni, K. et al. *BLEU: a Method for Automatic Evaluation of Machine
  Translation*.
- Popović, M. work on character n-gram evaluation and chrF.